# Redis Fundamentals — Hands-On

## Before You Start

Make sure Redis Stack is running:
```bash
docker run -d --name redis-stack \
  -p 6379:6379 -p 8001:8001 \
  redis/redis-stack:latest
```

Open RedisInsight at http://localhost:8001 to visually follow along.

In [2]:
# Install the Redis Python client
# redis-py is the official Redis client for Python
!uv pip install redis

Using Python 3.12.3 environment at: /home/hex/Documents/devops/dev-bits-daily/.venv
Resolved 1 package in 2.07s                                          
⠹ Preparing packages... (0/1)                                                   
⠹ Preparing packages... (0/1)-------------------     0 B/488.15 KiB          
⠹ Preparing packages... (0/1)------------------- 16.00 KiB/488.15 KiB        
⠸ Preparing packages... (0/1)------------------- 32.00 KiB/488.15 KiB        
⠸ Preparing packages... (0/1)------------------- 32.00 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 43.02 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 43.02 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 51.35 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 62.25 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 78.25 KiB/488.15 KiB        
⠼ Preparing packages... (0/1)------------------- 94.25 KiB/488.

## 1. Connect and Verify

In [3]:
import redis

# decode_responses=True: automatically decode bytes to strings
# Without it, Redis returns b'value' (bytes) instead of 'value' (str)
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# PING: the simplest health check
print("Connection:", r.ping())  # → True

# Server info
info = r.info()
print(f"Redis version: {info['redis_version']}")
print(f"Used memory: {info['used_memory_human']}")
print(f"Connected clients: {info['connected_clients']}")

Connection: True
Redis version: 7.4.7
Used memory: 1.47M
Connected clients: 3


## 2. Strings — SET / GET / INCR / SETEX

In [ ]:
# Basic set and get
r.set("greeting", "Hello from TechBot!")
print(r.get("greeting"))  # Hello from TechBot!

# Atomic integer counter — no race condition possible
r.set("counter:requests", 0)
r.incr("counter:requests")  # 1
r.incr("counter:requests")  # 2
r.incrby("counter:requests", 10)  # 12
print("Request counter:", r.get("counter:requests"))

# Set with TTL (expires in 10 seconds)
r.setex("temp:token", 10, "secret-api-token")
print("Token TTL:", r.ttl("temp:token"), "seconds")

# Only set if key does NOT exist (NX = Not eXists)
r.set("lock:inference", "worker-1", nx=True, ex=30)
r.set("lock:inference", "worker-2", nx=True, ex=30)  # fails silently
print("Lock holder:", r.get("lock:inference"))  # still worker-1

## 3. Hashes — Store Structured Objects

In [ ]:
import time

session_id = "user-alice-001"
meta_key = f"chat:{session_id}:metadata"

# HSET — set multiple fields at once
r.hset(meta_key, mapping={
    "user_id":       "user-alice",
    "model":         "llama3-8b",
    "system_prompt": "You are TechBot, a helpful support assistant.",
    "created_at":    str(int(time.time())),
    "message_count": "0"
})

# HGET — single field
print("Model:", r.hget(meta_key, "model"))

# HMGET — multiple fields at once
print("User + Model:", r.hmget(meta_key, "user_id", "model"))

# HGETALL — everything
print("Full metadata:", r.hgetall(meta_key))

# HINCRBY — atomically increment a numeric field
r.hincrby(meta_key, "message_count", 1)
r.hincrby(meta_key, "message_count", 1)
print("Message count:", r.hget(meta_key, "message_count"))

## 4. Lists — Chat Message Queue

In [ ]:
import json

messages_key = f"chat:{session_id}:messages"

# RPUSH — append to the right (chronological order)
def push_message(session_id, role, content):
    key = f"chat:{session_id}:messages"
    msg = json.dumps({
        "role": role,
        "content": content,
        "ts": int(time.time())
    })
    r.rpush(key, msg)
    r.ltrim(key, -100, -1)   # keep only last 100 messages
    r.expire(key, 86400)     # sliding 24-hour TTL

push_message(session_id, "user",      "How do I install the SDK?")
push_message(session_id, "assistant", "Run: pip install techbot-sdk")
push_message(session_id, "user",      "What about Windows?")
push_message(session_id, "assistant", "Same command — works on Windows too.")
push_message(session_id, "user",      "And how do I authenticate?")

# LLEN — count messages
print("Total messages:", r.llen(messages_key))

# LRANGE — fetch messages
# 0 -1 = all messages
# -3 -1 = last 3 messages
last_3 = r.lrange(messages_key, -3, -1)
print("\nLast 3 messages:")
for raw in last_3:
    msg = json.loads(raw)
    print(f"  [{msg['role']}]: {msg['content']}")

## 5. Sets — Session Tags and Feature Flags

In [ ]:
tags_key = f"session:{session_id}:tags"

# SADD — add members (duplicates silently ignored)
r.sadd(tags_key, "premium", "rag-enabled", "en", "beta-user")
r.sadd(tags_key, "premium")  # duplicate — no-op

# SISMEMBER — O(1) membership check
print("Is premium?", r.sismember(tags_key, "premium"))    # True
print("Is admin?",   r.sismember(tags_key, "admin"))      # False

# SMEMBERS — get all members (order not guaranteed)
print("All tags:", r.smembers(tags_key))

# SCARD — count members
print("Tag count:", r.scard(tags_key))

# SREM — remove a member
r.srem(tags_key, "beta-user")
print("After removing beta-user:", r.smembers(tags_key))

## 6. Sorted Sets — Priority Queue and Time-Ordered Log

In [ ]:
log_key = "inference:latency:log"

# ZADD — add members with scores
# Score here = unix timestamp of the inference call
now = int(time.time())
r.zadd(log_key, {"call:001": now - 300})   # 5 min ago
r.zadd(log_key, {"call:002": now - 200})   # 3 min ago
r.zadd(log_key, {"call:003": now - 100})   # 1 min ago
r.zadd(log_key, {"call:004": now})         # just now

# ZRANGE — fetch by rank (ascending)
print("All calls:", r.zrange(log_key, 0, -1, withscores=True))

# ZREVRANGE — fetch by rank (descending — most recent first)
print("Most recent first:", r.zrevrange(log_key, 0, 1, withscores=True))

# ZRANGEBYSCORE — fetch by score range (last 5 minutes)
five_min_ago = now - 300
recent = r.zrangebyscore(log_key, five_min_ago, "+inf", withscores=True)
print("Last 5 minutes:", recent)

# ZCARD — count members
print("Total log entries:", r.zcard(log_key))

# Priority queue pattern: lower score = higher priority
priority_queue = "tasks:priority"
r.zadd(priority_queue, {"task:low": 10, "task:med": 5, "task:high": 1})

# Pop the highest priority (lowest score)
highest = r.zpopmin(priority_queue, 1)
print("Highest priority task:", highest)

## 7. TTL / Expiry in Depth

In [ ]:
import time

# Create a key with a short TTL for demonstration
r.setex("demo:expiring", 5, "I will disappear in 5 seconds")

print("TTL right after creation:", r.ttl("demo:expiring"), "seconds")
print("Value:", r.get("demo:expiring"))

time.sleep(2)
print("\nAfter 2 seconds, TTL:", r.ttl("demo:expiring"), "seconds")

# Reset TTL (sliding window pattern)
r.expire("demo:expiring", 5)  # reset back to 5 seconds
print("After reset, TTL:", r.ttl("demo:expiring"), "seconds")

time.sleep(6)
result = r.get("demo:expiring")
print("\nAfter 6 more seconds:", result)  # None — key expired
print("TTL of expired key:", r.ttl("demo:expiring"))  # -2 means key doesn't exist

# TTL return values:
# -2 → key does not exist
# -1 → key exists but has no TTL (permanent)
# ≥ 0 → remaining seconds

## 8. Key Inspection and SCAN (never use KEYS * in production)

In [ ]:
# KEYS * — DO NOT use in production (blocks Redis during full keyspace scan)
# Safe here because we have few keys
all_keys = r.keys("chat:*")
print("Chat keys:", all_keys)

# SCAN — production-safe incremental scan
# Returns a cursor + a batch of keys. Repeat until cursor=0.
print("\nAll keys (via SCAN):")
for key in r.scan_iter("*"):
    key_type = r.type(key)
    print(f"  {key}  [{key_type}]")

## 9. Pipeline — Batch Commands for Speed

In [ ]:
import time

# Without pipeline: each command is a separate round-trip
start = time.perf_counter()
for i in range(100):
    r.set(f"bench:no_pipe:{i}", i)
no_pipe_time = time.perf_counter() - start

# With pipeline: all commands sent in one batch
start = time.perf_counter()
pipe = r.pipeline()
for i in range(100):
    pipe.set(f"bench:pipe:{i}", i)
pipe.execute()  # sends all 100 commands at once
pipe_time = time.perf_counter() - start

print(f"Without pipeline: {no_pipe_time*1000:.1f}ms")
print(f"With pipeline:    {pipe_time*1000:.1f}ms")
print(f"Speedup: {no_pipe_time/pipe_time:.1f}x")

## 10. Cleanup

In [ ]:
# Clean up all keys we created in this notebook
# In production you'd never call FLUSHDB — it wipes everything!
patterns_to_clean = [
    "greeting", "counter:requests", "temp:token", "lock:inference",
    "chat:user-alice-001:*", "session:user-alice-001:*",
    "inference:latency:log", "tasks:priority",
    "bench:no_pipe:*", "bench:pipe:*"
]

for pattern in patterns_to_clean:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)

print("Cleanup done. Remaining keys:", r.dbsize())

## Summary

You've now used all the core Redis data structures:

| Structure | Commands Used | TechBot Use |
|-----------|--------------|-------------|
| String | SET, GET, INCR, SETEX | Request counters, tokens |
| Hash | HSET, HGET, HGETALL | Session metadata |
| List | RPUSH, LRANGE, LTRIM | Chat message history |
| Set | SADD, SISMEMBER, SMEMBERS | Session tags, feature flags |
| Sorted Set | ZADD, ZRANGE, ZRANGEBYSCORE | Priority queues, time-ordered logs |

Next: **01_chat_history/02_hands_on.ipynb** — Build TechBot's memory layer.